# 04c · Seed Analysis — seed-to-seed variability, claim re-checks, CNN under-fitting flags (WP-B)

**Project:** Placement-specific axis reduction of wearable inertial sensors for gym exercise recognition (RecGym).

This notebook **trains nothing**. It reads the primary grid (seed 42) and the extra-seed runs and answers three questions that a reviewer is likely to ask about a single-seed study:

1. **How large is seed-to-seed variability** of the headline numbers, per model, placement and configuration?
2. **Do the paper's seed-sensitive claims survive other seeds?** Specifically "160 of 168 reductions are Holm-significant decreases" and "only calf gyro-3 meets the 0.05 non-inferiority margin in all four models".
3. **Which CNN units under-fit?** Every CNN unit with `final_train_loss` above 0.5 is flagged and counted by configuration, placement and seed.

> Every number comes from the real result files of your runs. If a seed is incomplete it is reported and **excluded** from the variability and test tables (a partial seed would bias the SD); `STRICT=1` turns an incomplete expected seed into an error.

---

## Inputs

| Seed(s) | Model(s) | Folder | Produced by |
|---|---|---|---|
| 42 | RF, SVM, XGB | `classical_full/` | notebook 01 (immutable) |
| 43, 44 | RF, SVM, XGB | `classical_seed43/`, `classical_seed44/` | notebook `04b_classical_seeds` |
| 42 | CNN | `cnn_full/` (primary CNN run; set `CNN_DIR` if it has another name) | notebook 02 (immutable) |
| 43 to 46 | CNN | `cnn_seeds43-46/full/` (found automatically, or set `CNN_EXTRA_DIRS`) | notebook `02_cnn_seeds43-46` |

Expected units per (model, seed): 3 positions × 10 folds × 15 configurations = **450**.

---

## Frozen analysis plan (fixed before any extra-seed result was seen)

| Item | Definition |
|---|---|
| Unit of analysis | Subject-level macro-F1 (12 labels incl. `Null`, over the classes present in the held-out subject), one value per (model, position, configuration, seed, fold). |
| Cell mean | Mean over the 10 held-out subjects, per seed. |
| Seed variability | Across seeds: mean, SD (ddof = 1), min, max, range of the cell mean. Also the mean over subjects of the within-subject SD across seeds. |
| Headline | 6-axis cells (3 positions × 4 models): mean ± SD across seeds. |
| Reduction test | Reduced configuration minus 6-axis, 10 paired subject-level values. Two-sided Wilcoxon signed-rank; percentile bootstrap 95 % CI over subjects (10,000 resamples, fixed generator seed); sign counts. **Holm within the primary family** (14 reduced configurations per position, model and seed). A *Holm-significant decrease* is p_Holm < 0.05 with a negative mean difference. |
| Non-inferiority margin (descriptive) | A reduced configuration **meets the margin** when the lower bound of the paired bootstrap 95 % CI of (reduced − 6-axis) is above −0.05. |
| Seed-averaged variant | Per model, subject-level values averaged over that model's complete seeds first, then the same tests (labelled seed = `mean`). |
| 160-of-168 re-check | Per seed, count of Holm-significant decreases over 14 configurations × 3 positions × 4 models (168 tests) for the seeds that exist for all four models; per model out of 42 otherwise. |
| Margin claim re-check | Per seed, the (position, configuration) pairs that meet the margin in **all** models available for that seed. |
| CNN under-fitting | `final_train_loss` > 0.5 is flagged. Epochs are **not** changed for the primary grid. |
| Not done here | No new family of tests, no selection of a best configuration, no energy claim. |

**What to expect from the numbers.** The primary results (notebook 03) stay the primary ones. Seeds are used to say how much those numbers move. The bootstrap here uses its own fixed random generator, so CI endpoints for seed 42 can differ from notebook 03's in the third decimal; a margin decision within 0.005 of the boundary is flagged (`near_boundary`) rather than hidden.

---

## Run control (environment variables or `RUN_OVERRIDES` in the next cell)

| Variable | Default | Meaning |
|---|---|---|
| `RAW_DIR` | `results/raw` | Folder holding `classical_full/`, `classical_seed43/`, `classical_seed44/`, `cnn_full/` (also searched under `<RAW_DIR>/results/raw`). |
| `CNN_DIR` | auto | Primary CNN run folder (seed 42), if it is not called `cnn_full`. |
| `CNN_EXTRA_DIRS` | auto | Comma-separated folders with the extra CNN seeds (each contains `results.csv`). Found automatically when the folder name contains `cnn` and `seed`. |
| `TABLES_DIR` | `results/tables_seeds` | Output folder for the CSVs, figure and manifest. |
| `STRICT` | unset | `1`: stop if an expected seed (classical 42, 43, 44; CNN 42 to 46) is missing or incomplete. |
| `N_BOOT` | `10000` | Bootstrap resamples. |

## 1 · Configuration and inputs

In [ ]:
import os, sys, re, json, time, hashlib, itertools, warnings
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon
warnings.filterwarnings("ignore")
try:
    display
except NameError:
    display = print

# ---- optional in-notebook overrides (Colab / Kaggle). Example: RUN_OVERRIDES = {"STRICT": "1", "CNN_DIR": "/content/cnn_merged"}
RUN_OVERRIDES = {"RAW_DIR": "/content/drive/MyDrive/Research/AxisFit-main"}
os.environ.update({str(k): str(v) for k, v in RUN_OVERRIDES.items()})

RAW_DIR = Path(os.environ.get("RAW_DIR", "results/raw"))
TABLES_DIR = Path(os.environ.get("TABLES_DIR", "results/tables_seeds"))
STRICT = os.environ.get("STRICT") == "1"
N_BOOT = int(os.environ.get("N_BOOT", "10000"))
BOOT_RNG_SEED = 20260101          # fixed generator seed of the bootstrap (does not interact with the experiment seeds)
MARGIN = 0.05                     # descriptive non-inferiority margin (macro-F1)
UNDERFIT_LOSS = 0.5               # CNN units with final_train_loss above this are flagged
ALPHA = 0.05
CLASSICAL_MODELS = ["RF", "SVM", "XGB"]
EXPECTED_SEEDS = {"classical": [42, 43, 44], "CNN": [42, 43, 44, 45, 46]}
N_FOLDS = 10

TABLES_DIR.mkdir(parents=True, exist_ok=True)
(TABLES_DIR / "figures").mkdir(parents=True, exist_ok=True)
LOG = []
def log(*a):
    s = " ".join(str(x) for x in a); print(s); LOG.append(s)

IMU = ["Ax", "Ay", "Az", "Gx", "Gy", "Gz"]
acc, gyr = IMU[:3], IMU[3:]
PRIMARY_CONFIGS = {"6-axis": list(IMU), "acc-3": list(acc), "gyro-3": list(gyr)}
for a, b in itertools.combinations(acc, 2): PRIMARY_CONFIGS[f"acc-{a}{b}"] = [a, b]
for a, b in itertools.combinations(gyr, 2): PRIMARY_CONFIGS[f"gyro-{a}{b}"] = [a, b]
for k in IMU: PRIMARY_CONFIGS[f"single-{k}"] = [k]
assert len(PRIMARY_CONFIGS) == 15
NCH = {c: len(v) for c, v in PRIMARY_CONFIGS.items()}
REDUCED = [c for c in PRIMARY_CONFIGS if c != "6-axis"]
assert len(REDUCED) == 14

def roots():
    return [RAW_DIR, RAW_DIR / "results" / "raw"]
def find_run(name):
    for b in roots():
        if (b / name / "results.csv").exists():
            return b / name
    return None

CLASSICAL_DIRS = {}
_p42 = find_run("classical_full")
assert _p42 is not None, f"classical_full/results.csv not found under {roots()}. Set RAW_DIR."
CLASSICAL_DIRS[42] = _p42
for b in roots():
    if b.exists():
        for d in sorted(b.glob("classical_seed*")):
            mm = re.fullmatch(r"classical_seed(\d+)", d.name)
            if mm and (d / "results.csv").exists():
                CLASSICAL_DIRS[int(mm.group(1))] = d
CNN_DIR = Path(os.environ["CNN_DIR"]) if os.environ.get("CNN_DIR") else find_run("cnn_full")
assert CNN_DIR is not None and (CNN_DIR / "results.csv").exists(), f"cnn_full/results.csv not found under {roots()}. Set RAW_DIR or CNN_DIR."
# extra CNN seeds live in their own folder (notebook 02_cnn_seeds43-46 writes <folder>/full/results.csv); found automatically,
# or listed explicitly in CNN_EXTRA_DIRS (comma-separated folders that contain results.csv)
CNN_EXTRA_DIRS = [Path(x.strip()) for x in os.environ.get("CNN_EXTRA_DIRS", "").split(",") if x.strip()]
if not CNN_EXTRA_DIRS:
    for b in roots():
        if b.exists():
            for d in sorted(b.glob("*cnn*seed*")):
                if d.is_dir() and "smoke" not in d.name.lower():
                    for cand in (d / "full", d):
                        if (cand / "results.csv").exists():
                            CNN_EXTRA_DIRS.append(cand); break
for d in CNN_EXTRA_DIRS:
    assert (d / "results.csv").exists(), f"CNN_EXTRA_DIRS entry has no results.csv: {d}"
log("Classical runs:", {s: str(d) for s, d in sorted(CLASSICAL_DIRS.items())})
log("CNN run (primary):", CNN_DIR, "| extra CNN seed folders:", [str(d) for d in CNN_EXTRA_DIRS] or "none found")

KEEP = ["position", "config", "model", "fold", "seed", "macro_f1"]
frames = []
for s, d in sorted(CLASSICAL_DIRS.items()):
    df = pd.read_csv(d / "results.csv")
    if "seed" not in df.columns:
        assert s == 42, f"{d}/results.csv has no seed column but is not the seed-42 run"
        df["seed"] = 42
    df = df[df["model"].isin(CLASSICAL_MODELS)]
    if s == 42:
        df = df[df["seed"] == 42]
    assert (df["seed"] == s).all(), f"{d}: rows with a seed other than {s}"
    frames.append(df[KEEP])
cnn_parts = []
for d in [CNN_DIR] + CNN_EXTRA_DIRS:
    part = pd.read_csv(d / "results.csv")
    assert "seed" in part.columns and "final_train_loss" in part.columns, f"{d}/results.csv lacks seed / final_train_loss"
    part = part[part["model"] == "CNN"].copy()
    part["_src"] = str(d)
    log("CNN file", d, "| rows:", len(part), "| seeds:", sorted(int(x) for x in part["seed"].unique()))
    cnn_parts.append(part)
cnn_raw = pd.concat(cnn_parts, ignore_index=True)
_dup = cnn_raw.duplicated(subset=["position", "config", "model", "fold", "seed"], keep=False)
assert not _dup.any(), ("the same CNN unit appears in more than one folder (" + str(int(_dup.sum())) + " rows). "
                        "Each seed must live in exactly one folder; seed 42 only in the primary folder.")
# same frozen design in all CNN folders? (data hash, window hash, architecture, training settings)
def _design(d):
    f = d / "run_config.json"
    return json.load(open(f)).get("design", {}) if f.exists() else None
_d0 = _design(CNN_DIR)
for d in CNN_EXTRA_DIRS:
    _d1 = _design(d)
    if _d0 is None or _d1 is None:
        log("Note: run_config.json missing in", CNN_DIR if _d0 is None else d, "; design not compared.")
        continue
    diff = [k for k in ("data_sha256", "windows_sha256", "cnn", "train") if _d0.get(k) != _d1.get(k)]
    if diff:
        log("WARNING: design differs between", CNN_DIR, "and", d, "in", diff, "-> the seeds are NOT comparable until this is explained.")
        if STRICT:
            raise AssertionError(f"STRICT=1 and the CNN design differs in {diff}")
    else:
        log("Design check OK:", d.name, "has the same data hash, window hash, architecture and training settings as the primary CNN run.")
frames.append(cnn_raw[KEEP])
long = pd.concat(frames, ignore_index=True)
long["seed"] = long["seed"].astype(int); long["fold"] = long["fold"].astype(int)

# ---- integrity
assert not long.duplicated(subset=["position", "config", "model", "fold", "seed"]).any(), "duplicated units"
assert set(long["config"]) <= set(PRIMARY_CONFIGS), f"configurations outside the primary 15: {sorted(set(long['config']) - set(PRIMARY_CONFIGS))}"
assert long["macro_f1"].between(0, 1).all() and long["macro_f1"].notna().all(), "macro_f1 outside [0, 1] or missing"
POSITIONS = sorted(long["position"].unique())
assert POSITIONS == ["leg", "pocket", "wrist"], POSITIONS
assert set(long["fold"]) == set(range(N_FOLDS)), sorted(set(long["fold"]))
N_EXPECTED = len(POSITIONS) * N_FOLDS * len(PRIMARY_CONFIGS)       # 450
MODELS = [m for m in ["RF", "SVM", "XGB", "CNN"] if m in set(long["model"])]
log("Models:", MODELS, "| positions:", POSITIONS, "| expected units per (model, seed):", N_EXPECTED)

# ---- completeness per (model, seed); only complete seeds are used below
comp = long.groupby(["model", "seed"]).size().reset_index(name="n_units")
comp["expected"] = N_EXPECTED
comp["complete"] = comp["n_units"] == comp["expected"]
comp["missing"] = comp["expected"] - comp["n_units"]
exp_rows = []
for m in MODELS:
    for s in EXPECTED_SEEDS["CNN" if m == "CNN" else "classical"]:
        if not ((comp["model"] == m) & (comp["seed"] == s)).any():
            exp_rows.append(dict(model=m, seed=s, n_units=0, expected=N_EXPECTED, complete=False, missing=N_EXPECTED))
if exp_rows:
    comp = pd.concat([comp, pd.DataFrame(exp_rows)], ignore_index=True)
comp = comp.sort_values(["model", "seed"]).reset_index(drop=True)
comp.to_csv(TABLES_DIR / "seed_completeness.csv", index=False)
display(comp)
_bad = comp[(~comp["complete"]) & comp.apply(lambda r: r["seed"] in EXPECTED_SEEDS["CNN" if r["model"] == "CNN" else "classical"], axis=1)]
if len(_bad):
    log("INCOMPLETE OR MISSING expected seeds (excluded from the analysis):", [(r.model, int(r.seed), int(r.n_units)) for r in _bad.itertuples()])
    if STRICT:
        raise AssertionError("STRICT=1 and an expected seed is missing or incomplete (see seed_completeness.csv)")
SEEDS_USED = {m: sorted(int(s) for s in comp[(comp["model"] == m) & comp["complete"]]["seed"]) for m in MODELS}
log("Complete seeds used:", SEEDS_USED)
used = long[long.apply(lambda r: r["seed"] in SEEDS_USED[r["model"]], axis=1)].copy()
used["n_channels"] = used["config"].map(NCH)
assert len(used) == sum(len(v) for v in SEEDS_USED.values()) * N_EXPECTED
MODELS_VAR = [m for m in MODELS if len(SEEDS_USED[m]) >= 2]
if set(MODELS_VAR) != set(MODELS):
    log("Models with fewer than 2 complete seeds (no SD possible):", sorted(set(MODELS) - set(MODELS_VAR)))

## 2 · Seed-to-seed variability of the cell means

For every (model, position, configuration): the cell mean (mean over the 10 held-out subjects) is computed per seed; across seeds the mean, SD (ddof = 1), min, max and range are reported. `mean_within_subject_seed_sd` is the SD across seeds of the subject-level value, averaged over the 10 subjects: it measures the seed noise a single subject's number carries.

In [ ]:
cell = used.groupby(["model", "position", "config", "seed"])["macro_f1"].mean().reset_index(name="cell_mean")
def _sd(x): return float(np.std(np.asarray(x, dtype=float), ddof=1)) if len(x) > 1 else np.nan
var = (cell.groupby(["model", "position", "config"])["cell_mean"]
       .agg(n_seeds="count", mean_over_seeds="mean", sd_over_seeds=_sd, min_over_seeds="min", max_over_seeds="max").reset_index())
var["range_over_seeds"] = var["max_over_seeds"] - var["min_over_seeds"]
var["n_channels"] = var["config"].map(NCH)

fold_sd = (used.groupby(["model", "position", "config", "fold"])["macro_f1"].agg(_sd).reset_index(name="sd_fold")
           .groupby(["model", "position", "config"])["sd_fold"].mean().reset_index(name="mean_within_subject_seed_sd"))
var = var.merge(fold_sd, on=["model", "position", "config"], how="left")
var = var[var["model"].isin(MODELS_VAR)].sort_values(["model", "position", "n_channels", "config"], ascending=[True, True, False, True]).reset_index(drop=True)
var.to_csv(TABLES_DIR / "seed_variability_cells.csv", index=False)
log("Cells with seed variability:", len(var), "(expected", len(MODELS_VAR) * len(POSITIONS) * 15, ")")
assert len(var) == len(MODELS_VAR) * len(POSITIONS) * 15

# ---- headline: 6-axis cells, mean +- SD across seeds, plus the per-seed values
six = var[var["config"] == "6-axis"].copy()
wide = cell[cell["config"] == "6-axis"].pivot_table(index=["model", "position"], columns="seed", values="cell_mean").add_prefix("seed_").reset_index()
head = six[["model", "position", "n_seeds", "mean_over_seeds", "sd_over_seeds", "min_over_seeds", "max_over_seeds", "range_over_seeds"]].merge(wide, on=["model", "position"], how="left")
head["mean_pm_sd"] = head.apply(lambda r: f"{r['mean_over_seeds']:.3f} ± {r['sd_over_seeds']:.3f}", axis=1)
head.to_csv(TABLES_DIR / "seed_headline_6axis.csv", index=False)
display(head.round(4))

# ---- how large is the SD overall, by model and channel count
by_ch = (var.groupby(["model", "n_channels"])["sd_over_seeds"]
         .agg(n_cells="count", median_sd="median", mean_sd="mean", max_sd="max").reset_index())
by_ch.to_csv(TABLES_DIR / "seed_sd_by_model_channels.csv", index=False)
by_model = var.groupby("model")["sd_over_seeds"].agg(n_cells="count", median_sd="median", mean_sd="mean", max_sd="max").reset_index()
by_model.to_csv(TABLES_DIR / "seed_sd_by_model.csv", index=False)
display(by_model.round(4)); display(by_ch.round(4))
worst = var.sort_values("sd_over_seeds", ascending=False).head(10)[["model", "position", "config", "n_seeds", "mean_over_seeds", "sd_over_seeds", "range_over_seeds"]]
log("Ten cells with the largest seed SD:"); display(worst.round(4))

## 3 · Re-checking the primary-family claims under every seed

For each model, position and seed (and for the seed-averaged variant), the 14 reduced configurations are compared with 6-axis on the 10 paired subject-level values. Holm is applied **within the primary family** (14 tests per position, model and seed), exactly as in the primary analysis. Two claims are then re-counted:

* the number of Holm-significant decreases (the "160 of 168" statement), and
* the (position, configuration) pairs that meet the 0.05 non-inferiority margin in all models available for that seed (the "only calf gyro-3" statement).

Wilcoxon uses `scipy.stats.wilcoxon` (two-sided; exact for n = 10 without ties); the bootstrap is a percentile bootstrap over subjects.

In [ ]:
VEC = {}
for (m, p, c, s), g in used.groupby(["model", "position", "config", "seed"]):
    g = g.sort_values("fold")
    assert list(g["fold"]) == list(range(N_FOLDS)), f"{(m, p, c, s)}: folds {list(g['fold'])}"
    VEC[(m, p, c, s)] = g["macro_f1"].to_numpy(dtype=float)
for (m, p, c), g in used.groupby(["model", "position", "config"]):
    VEC[(m, p, c, "mean")] = np.vstack([VEC[(m, p, c, s)] for s in SEEDS_USED[m]]).mean(axis=0)

def boot_ci(d, n_boot=N_BOOT, rng_seed=BOOT_RNG_SEED):
    rng = np.random.default_rng(rng_seed)
    idx = rng.integers(0, len(d), size=(n_boot, len(d)))
    means = d[idx].mean(axis=1)
    lo, hi = np.percentile(means, [2.5, 97.5])
    return float(lo), float(hi)

def wilcoxon_p(d):
    if np.allclose(d, 0.0):
        return 1.0
    try:
        return float(wilcoxon(d, alternative="two-sided").pvalue)
    except ValueError:
        return 1.0

def holm(p):
    p = np.asarray(p, dtype=float); n = len(p); order = np.argsort(p); adj = np.empty(n); run = 0.0
    for rank, i in enumerate(order):
        run = max(run, (n - rank) * p[i]); adj[i] = min(1.0, run)
    return adj

MODELS_T = [m for m in MODELS if SEEDS_USED[m]]
if not MODELS_T:
    raise AssertionError("no model has a complete seed: nothing to test")
rows = []
for m in MODELS_T:
    for p in POSITIONS:
        for s in list(SEEDS_USED[m]) + ["mean"]:
            base = VEC[(m, p, "6-axis", s)]
            recs = []
            for c in REDUCED:
                d = VEC[(m, p, c, s)] - base
                lo, hi = boot_ci(d)
                recs.append(dict(model=m, position=p, config=c, seed=str(s), n_channels=NCH[c], mean_diff=float(d.mean()), ci_lo=lo, ci_hi=hi,
                                 p=wilcoxon_p(d), n_subjects_lower=int((d < 0).sum())))
            ph = holm([r["p"] for r in recs])
            for r, x in zip(recs, ph):
                r["p_holm"] = float(x)
            rows += recs
tests = pd.DataFrame(rows)
tests["sig_decrease"] = (tests["p_holm"] < ALPHA) & (tests["mean_diff"] < 0)
tests["meets_margin"] = tests["ci_lo"] > -MARGIN
tests["near_boundary"] = (tests["ci_lo"] + MARGIN).abs() < 0.005
assert len(tests) == len(POSITIONS) * 14 * sum(len(SEEDS_USED[m]) + 1 for m in MODELS_T), "unexpected number of reduction tests"
tests.to_csv(TABLES_DIR / "reduction_tests_by_seed.csv", index=False)
log("Reduction tests:", len(tests), "rows | families of 14 (Holm) =", len(tests) // 14)

# ---- 160-of-168 re-check: counts of Holm-significant decreases per model and seed
cnt = (tests.groupby(["model", "seed"]).agg(n_tests=("p", "count"), n_sig_decrease=("sig_decrease", "sum"), n_meet_margin=("meets_margin", "sum")).reset_index())
cnt["n_sig_decrease"] = cnt["n_sig_decrease"].astype(int); cnt["n_meet_margin"] = cnt["n_meet_margin"].astype(int)
# all-model rows for seeds that exist for every model (the "of 168" statement)
common = set.intersection(*[set(SEEDS_USED[m]) for m in MODELS_T]) if len(MODELS_T) == 4 else set()
allrows = []
for s in sorted(common) + ["mean"]:
    q = tests[tests["seed"] == str(s)]
    allrows.append(dict(model="ALL (" + "+".join(MODELS_T) + ")", seed=str(s), n_tests=len(q), n_sig_decrease=int(q["sig_decrease"].sum()), n_meet_margin=int(q["meets_margin"].sum())))
cnt = pd.concat([cnt, pd.DataFrame(allrows)], ignore_index=True)
cnt.to_csv(TABLES_DIR / "holm_counts_by_seed.csv", index=False)
display(cnt)
for r in allrows:
    log(f"Seed {r['seed']}: {r['n_sig_decrease']} of {r['n_tests']} reductions are Holm-significant decreases; {r['n_meet_margin']} meet the 0.05 margin")

# ---- which tests are NOT significant, per seed (are they the same ones?)
ns = tests[(~tests["sig_decrease"]) & (tests["seed"].isin([str(s) for s in sorted(common)] + ["mean"]))]
ns[["model", "position", "config", "seed", "mean_diff", "ci_lo", "ci_hi", "p_holm", "n_subjects_lower"]].sort_values(["seed", "model", "position", "config"]).to_csv(
    TABLES_DIR / "not_significant_reductions_by_seed.csv", index=False)
if len(ns):
    agree = (ns.groupby(["model", "position", "config"])["seed"].apply(lambda x: ",".join(sorted(x))).reset_index(name="seeds_not_significant"))
else:
    agree = pd.DataFrame(columns=["model", "position", "config", "seeds_not_significant"])
agree.to_csv(TABLES_DIR / "not_significant_agreement_across_seeds.csv", index=False)
log("Reductions that are not Holm-significant decreases in at least one seed:", len(agree)); display(agree)

# ---- margin claim: pairs meeting the margin in all models available for that seed
mc_rows = []
for (sd_, pos_, cfg_), g in tests.groupby(["seed", "position", "config"]):
    meet = g[g["meets_margin"]]
    mc_rows.append(dict(seed=sd_, position=pos_, config=cfg_, n_models=int(len(g)), n_models_meeting=int(len(meet)),
                        models_meeting="+".join(sorted(meet["model"])), meets_in_all_models=bool(len(meet) == len(g)),
                        any_near_boundary=bool(g["near_boundary"].any())))
mc = pd.DataFrame(mc_rows)
mc.to_csv(TABLES_DIR / "margin_claim_by_seed.csv", index=False)
claim_rows = []
for s in sorted(mc["seed"].unique(), key=lambda x: (x == "mean", x)):
    q = mc[(mc["seed"] == s) & mc["meets_in_all_models"]]
    nm = int(mc[mc["seed"] == s]["n_models"].max())
    pairs = sorted(f"{r.position}:{r.config}" for r in q.itertuples())
    claim_rows.append(dict(seed=s, n_models_for_seed=nm, pairs_meeting_margin_in_all_models="; ".join(pairs) if pairs else "(none)",
                           n_pairs=len(pairs), any_pair_near_boundary=bool(q["any_near_boundary"].any())))
claims = pd.DataFrame(claim_rows)
claims.to_csv(TABLES_DIR / "margin_claim_summary.csv", index=False)
display(claims)
for r in claim_rows:
    log(f"Seed {r['seed']} ({r['n_models_for_seed']} model(s)): pairs meeting the 0.05 margin in all models -> {r['pairs_meeting_margin_in_all_models']}")
_c42 = [r for r in claim_rows if r["seed"] == "42"]
if _c42 and len(MODELS_T) == 4:
    same = _c42[0]["pairs_meeting_margin_in_all_models"] == "leg:gyro-3"
    log("Seed 42 recomputed here reproduces 'only leg:gyro-3 meets the margin in all four models':", same,
        "(a difference near the boundary can come from the bootstrap generator; see near_boundary in margin_claim_by_seed.csv)")

## 4 · CNN under-fitting flags

Every CNN unit (all seeds present in the file, complete or not) with `final_train_loss` above 0.5 is flagged. The primary grid keeps 25 epochs; an optional appendix with more epochs on flagged settings is a separate, clearly labelled run and is not part of this notebook.

In [ ]:
cn = cnn_raw[["position", "config", "model", "fold", "seed", "macro_f1", "final_train_loss"]].copy()
assert cn["final_train_loss"].notna().all(), "CNN units without final_train_loss"
cn["n_channels"] = cn["config"].map(NCH)
cn["flagged"] = cn["final_train_loss"] > UNDERFIT_LOSS
cn["flagged_i"] = cn["flagged"].astype(int)
log("CNN units:", len(cn), "| seeds:", sorted(cn["seed"].unique()), "| flagged (final_train_loss >", UNDERFIT_LOSS, "):", int(cn["flagged"].sum()))

cn[cn["flagged"]].sort_values(["seed", "position", "config", "fold"]).to_csv(TABLES_DIR / "cnn_underfit_units.csv", index=False)

by_seed = cn.groupby("seed").agg(n_units=("flagged", "count"), n_flagged=("flagged", "sum"), max_final_train_loss=("final_train_loss", "max"),
                                 median_final_train_loss=("final_train_loss", "median")).reset_index()
by_seed["share_flagged"] = by_seed["n_flagged"] / by_seed["n_units"]
by_seed.to_csv(TABLES_DIR / "cnn_underfit_by_seed.csv", index=False)
display(by_seed.round(4))

by_cfg = cn.groupby(["position", "config", "n_channels"]).agg(n_units=("flagged", "count"), n_flagged=("flagged", "sum"),
                                                              mean_final_train_loss=("final_train_loss", "mean"), max_final_train_loss=("final_train_loss", "max")).reset_index()
by_cfg["share_flagged"] = by_cfg["n_flagged"] / by_cfg["n_units"]
per_seed = cn.pivot_table(index=["position", "config"], columns="seed", values="flagged_i", aggfunc="sum").add_prefix("flagged_seed_").reset_index()
by_cfg = by_cfg.merge(per_seed, on=["position", "config"], how="left").sort_values(["n_flagged", "mean_final_train_loss"], ascending=False)
by_cfg.to_csv(TABLES_DIR / "cnn_underfit_by_config.csv", index=False)
flagged_cfg = by_cfg[by_cfg["n_flagged"] > 0]
log("(position, configuration) cells with at least one flagged unit:", len(flagged_cfg), "of", len(by_cfg))
display(flagged_cfg.round(4))

cn_by_ch = cn.groupby("n_channels").agg(n_units=("flagged", "count"), n_flagged=("flagged", "sum")).reset_index()
cn_by_ch["share_flagged"] = cn_by_ch["n_flagged"] / cn_by_ch["n_units"]
cn_by_ch.to_csv(TABLES_DIR / "cnn_underfit_by_channels.csv", index=False)
display(cn_by_ch.round(4))
log("Largest final_train_loss:", round(float(cn["final_train_loss"].max()), 3), "| flagged units by channel count:", {int(r.n_channels): int(r.n_flagged) for r in cn_by_ch.itertuples()})

## 5 · Figure: seed SD against channel count

In [ ]:
import matplotlib.pyplot as plt

colors = {"leg": "#1b9e77", "pocket": "#d95f02", "wrist": "#7570b3"}
fig, axes = plt.subplots(1, len(MODELS_VAR), figsize=(3.2 * len(MODELS_VAR), 3.2), sharey=True)
axes = np.atleast_1d(axes)
rng = np.random.default_rng(0)
for ax, m in zip(axes, MODELS_VAR):
    q = var[var["model"] == m]
    for p in POSITIONS:
        qq = q[q["position"] == p]
        ax.scatter(qq["n_channels"] + rng.uniform(-0.12, 0.12, len(qq)), qq["sd_over_seeds"], s=16, alpha=0.8, color=colors[p], label=p)
    ax.set_title(f"{m} ({len(SEEDS_USED[m])} seeds)", fontsize=9)
    ax.set_xlabel("channels"); ax.set_xticks([1, 2, 3, 6]); ax.grid(alpha=0.25)
axes[0].set_ylabel("SD of cell-mean macro-F1 across seeds")
axes[0].legend(fontsize=7, frameon=False)
fig.tight_layout()
for ext in ("png", "pdf"):
    fig.savefig(TABLES_DIR / "figures" / f"F_seed_sd_by_channels.{ext}", dpi=300, bbox_inches="tight")
plt.show()

## 6 · Manifest

Every file written by this notebook, with row counts and SHA-256, plus the input files and the frozen parameters, so each number quoted in the manuscript can be traced to a file and a row.

In [ ]:
def sha(path, chunk=1 << 22):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b: break
            h.update(b)
    return h.hexdigest()

outputs = []
for f in sorted(TABLES_DIR.rglob("*")):
    if f.is_file() and f.name != "seed_analysis_manifest.json":
        rec = {"file": str(f.relative_to(TABLES_DIR)), "sha256": sha(f)}
        if f.suffix == ".csv":
            rec["rows"] = int(len(pd.read_csv(f)))
        outputs.append(rec)
inputs = [{"file": str(d / "results.csv"), "role": f"classical seed {s}", "sha256": sha(d / "results.csv")} for s, d in sorted(CLASSICAL_DIRS.items())]
inputs.append({"file": str(CNN_DIR / "results.csv"), "role": "CNN primary folder", "sha256": sha(CNN_DIR / "results.csv")})
for d in CNN_EXTRA_DIRS:
    inputs.append({"file": str(d / "results.csv"), "role": "CNN extra seeds", "sha256": sha(d / "results.csv")})
manifest = {"notebook": "04c_seed_analysis", "timestamp_utc": pd.Timestamp.now("UTC").isoformat(),
            "parameters": {"N_BOOT": N_BOOT, "BOOT_RNG_SEED": BOOT_RNG_SEED, "MARGIN": MARGIN, "UNDERFIT_LOSS": UNDERFIT_LOSS, "ALPHA": ALPHA,
                           "holm_family": "14 reduced configurations per position, model and seed", "STRICT": STRICT},
            "seeds_used": {m: SEEDS_USED[m] for m in MODELS}, "expected_seeds": EXPECTED_SEEDS,
            "packages": {"numpy": np.__version__, "pandas": pd.__version__, "scipy": __import__("scipy").__version__},
            "inputs": inputs, "outputs": outputs}
json.dump(manifest, open(TABLES_DIR / "seed_analysis_manifest.json", "w"), indent=2)
with open(TABLES_DIR / "seed_analysis_log.txt", "w") as f:
    f.write("\n".join(LOG) + "\n")
log("Manifest written:", TABLES_DIR / "seed_analysis_manifest.json", "|", len(outputs), "output files")